# A0. Mechanisms, Ions, and Materials

In this tutorial, we dive into the nitty gritty of how mechanisms are implemented in Dendra. We will explore how to define custom ion channels, local state dynamics, shared material fields, material processes such as diffusion/clearance/clamping/exchange, and synaptic mechanisms, and how to integrate them into neuronal models.

## Mechanism, State, Material, and MaterialProcess

Dendra separates local biophysics from shared population-wide fields.

- **Mechanism** (`Mechanism`): user-facing local biophysics. A mechanism declares parameters, state bundles, currents, ion/material dependencies, and lifecycle hooks.
- **State** (`State`): helper class for private per-mechanism state variables and their ODEs/kinetics. Register `State` subclasses with `Mechanism.STATE(...)`.
- **Material**: population-wide shared fields that exist in every compartment and can be read/written/sourced by multiple mechanisms. Ions are specialized materials with valence, current, and Nernst/reversal-potential semantics.
- **MaterialProcess**: process-level dynamics over full material fields. Examples include `DiffusionProcess`, `ClearanceProcess`, `ClampProcess`, and `ExchangeProcess`. These are inserted like mechanisms but are scheduled in material phases rather than treated as ordinary local channel mechanisms.

A useful rule of thumb is: use `State` for variables private to one mechanism; use `Material` when the variable represents shared biochemical, chemical, or ionic state that several mechanisms may need to read, write, source, or spatially process.

### Membrane-mechanism unit contract

Dendra uses plain numeric tensors rather than unit-carrying quantities, so mechanism authors are responsible for following the public unit contract:

- An ordinary `Mechanism` (including `ContinuousSynapse`) is **distributed** over membrane area. Voltage and reversal potentials are in mV; current methods return outward-positive current density in mA/cm²; conductance parameters and `dI/dV` are in S/cm². The identity S × mV = mA makes `g * (v - e)` a current density without an extra factor of 1000.
- A `PointProcess` is **lumped** at a finite-area compartment. Its current methods return nA and its conductances use µS; Dendra divides the raw values by `1e6 * area_cm2` to obtain mA/cm² and S/cm² for the density-based cable equation. The corresponding identity is µS × mV = nA. A point process therefore cannot be attached to a zero-area branchpoint.

Do not mix these contracts: a distributed channel parameter such as HH `gnabar=0.12` means 0.12 S/cm² (120 mS/cm²), whereas a point-synapse conductance of `0.001` means 0.001 µS (1 nS). Built-in point-synapse weights are bare values in this local µS coordinate and must not be multiplied by `dendra.units.uS`. See {doc}`../units` for the complete convention.


## Declaring state variables and dynamics

Use a `State` subclass to name state variables and define their dynamics with `DERIVATIVE` or `KINETIC`. `breakpoint` computes `ASSIGNED` variables (steady states, time constants, etc.) that appear in derivatives/currents but are not themselves state or parameter variables.

Simulation time and `dt` are in ms. Consequently, if a state variable `x` has units U, the right-hand side of `x' = ...` must evaluate in U/ms; a first-order rate constant is in ms⁻¹ and a time constant is in ms. Dendra does not infer or validate these dimensions. For an SDE, the diffusion coefficient multiplying `dW` has units U/√ms.


In [ ]:
from dendra.models.mechanisms import Mechanism as M
from dendra.models.mechanisms import State as S
from dendra.models.mechanisms import (
    DiffusionProcess,
    ClearanceProcess,
    ClampProcess,
    ExchangeProcess,
    register_material,
    register_ion,
)
from dendra.models.mechanisms.ops import exp, vtrap

class mhn(S):
    has_q10 = True  # optional Q10 scaling hook

    S.STATE("m", "h", "n")
    S.METHOD("cnexp")
    S.DERIVATIVE(
        "m' = (minf - m) / mtau",
        "h' = (hinf - h) / htau",
        "n' = (ninf - n) / ntau",
    )
    S.ASSIGNED("minf", "mtau", "hinf", "htau", "ninf", "ntau")

    def calc_q10(self):
        return 3.0 ** ((self.celsius - 6.3) / 10.0)

    def breakpoint(self, v, states=None):
        q10 = self.q10()
        alpha_m = 0.1 * vtrap(-(v + 40), 10)
        beta_m = 4 * exp(-(v + 65) / 18)
        tot = alpha_m + beta_m
        mtau = 1 / (q10 * tot)
        minf = alpha_m / tot

        alpha_h = 0.07 * exp(-(v + 65) / 20)
        beta_h = 1 / (exp(-(v + 35) / 10) + 1)
        tot = alpha_h + beta_h
        htau = 1 / (q10 * tot)
        hinf = alpha_h / tot

        alpha_n = 0.01 * vtrap(-(v + 55), 10)
        beta_n = 0.125 * exp(-(v + 65) / 80)
        tot = alpha_n + beta_n
        ntau = 1 / (q10 * tot)
        ninf = alpha_n / tot

        # return dict whose keys match ASSIGNED names
        return {"mtau": mtau, "minf": minf, "htau": htau, "hinf": hinf, "ntau": ntau, "ninf": ninf}

    def inf(self, v):
        states = self.breakpoint(v, None)
        return {"m": states["minf"], "h": states["hinf"], "n": states["ninf"]}


class hh(M):
    """
    Classic Hodgkin-Huxley mechanism with m/h/n gates and leak.
    """

    M.STATE(mhn)
    # Conductance densities are in S/cm²; reversal potentials are in mV.
    M.GLOBAL(gnabar=0.12, gkbar=0.036, gl=0.0003, ena=50.0, ek=-77.0, el=-54.3)
    M.NONSPECIFIC_CURRENT("il", "ina", "ik")
    # Example for ions: M.USEION("na", read=("nai", "nao", "ena"), write=("ina",))

    def il(self, v):
        return self.gl * (v - self.el)

    def ina(self, v):
        gna = self.gnabar * self.m**3 * self.h
        return gna * (v - self.ena)

    def ik(self, v):
        gk = self.gkbar * self.n**4
        return gk * (v - self.ek)


## Methods of integration

Dendra supports multiple methods for integrating `State` dynamics. The preferred way to select an integration method is with the `State.METHOD(...)` declaration inside a `State` subclass:

```python
from dendra.models.mechanisms._state import State as S

class MyState(S):
    S.STATE("x", "y")
    S.METHOD("rosenbrock", fallback="derivimplicit")

    S.DERIVATIVE(
        "x' = -a*x + y*y",
        "y' = -b*y + x",
    )
```

`S.METHOD(method, **kwargs)` selects the integration method and passes method-specific options to the solver builder. The older class-attribute form is still supported for backward compatibility:

```python
class MyState(S):
    method = "derivimplicit"
```

However, `S.METHOD(...)` is preferred because it also supports method options such as iteration counts, damping, fallbacks, or solver-specific declarations.


### `"cnexp"`

Exact exponential integration for independent scalar affine ODEs.

Use this for equations of the form:

```text
x' = a + b*x
```

or, equivalently, common gating equations such as:

```text
x' = (x_inf - x) / tau_x
```

where coefficients are treated as fixed over the timestep.

Example:

```python
class Gate(S):
    S.STATE("m")
    S.METHOD("cnexp")

    S.DERIVATIVE(
        "m' = (minf - m) / mtau",
    )
```

`"cnexp"` is the default method. It is usually the fastest and most accurate method for Hodgkin-Huxley-style independent gates. It should be preferred whenever the state equation is independent and affine in its own state.

`cnexp` can also be applied to systems where each scalar equation is affine in its own state but depends on other state variables. In that case, the other states are treated as constants over the timestep, using their values at the beginning of the step. This is a frozen-coupling approximation: it is fast and often useful, but it is not an exact coupled solve.

Supported aliases include:

```text
"cnexp"
"rush_larsen"
"rl"
```


### `"bufferimplicit"`

Specialized implicit integration for two-state free/bound buffer systems.

Use this for systems like calcium buffering:

```text
bound' = ku * free * (1 - bound) - kr * bound
free'  = source - gamma * bound'
```

For example:

```python
class oc_cai(S):
    S.STATE("oc", "cai")
    S.METHOD("bufferimplicit", bound="oc", free="cai")

    S.DERIVATIVE(
        "oc' = ku * cai * (1 - oc) - kr * oc",
        "cai' = -ica * SA / Vol / (2*FARADAY) - nb * Bi * (ku * cai * (1 - oc) - kr * oc)",
    )
```

This method performs a closed-form backward-Euler update by reducing the implicit system to a scalar quadratic solve. It avoids Newton iteration, preserves the buffer/free-concentration conservation relation under the backward-Euler discretization, and is much faster than `derivimplicit` for this motif.

This is the preferred method for `oc/cai`-style intracellular calcium buffering systems.

Common options:

```python
S.METHOD(
    "bufferimplicit",
    bound="oc",
    free="cai",
    clamp_discriminant=True,
    eps=1e-30,
    clamp_bound=False,
    bound_min=0.0,
    bound_max=1.0,
    free_min=None,
    strict=True,
    fallback=None,
)
```

Supported aliases include:

```text
"bufferimplicit"
"buffer"
"implicit_buffer"
"ca_buffer"
"calcium_buffer"
"calciumbuffer"
```


### `"linearimplicit"` / `"sparse"`

Implicit backward-Euler integration for coupled affine-linear systems.

Use this for systems of the form:

```text
x' = A*x + b
```

where the state variables may be coupled, but the right-hand side is affine-linear in the state vector.

Example:

```python
class Markov2(S):
    S.STATE("C", "O")
    S.METHOD("sparse")

    S.DERIVATIVE(
        "C' = -alpha*C + beta*O",
        "O' =  alpha*C - beta*O",
    )
```

The method generates the backward-Euler update:

```text
(I - dt*A) * x_next = x + dt*b
```

This is the correct choice for coupled linear Markov models, linear kinetic schemes, linear exchange systems, and other affine-coupled state systems.

Although `"sparse"` is provided as an alias for NMODL familiarity, the current implementation is best thought of as a generated batched small-system linear implicit solver. It does not require the user to manually assemble a sparse matrix.

This method is not valid for nonlinear state coupling such as:

```text
x' = x*y
x' = sin(x) - y
oc' = ku*cai*(1 - oc) - kr*oc
```

For the last case, use `"bufferimplicit"` if the equations match the calcium-buffer motif, or `"rosenbrock"` / `"derivimplicit"` otherwise.

Common options:

```python
S.METHOD(
    "sparse",
    solver="auto",
    jacobian_regularization=0.0,
    strict=True,
    fallback=None,
)
```

Fallback examples:

```python
S.METHOD("sparse", fallback="rosenbrock")
S.METHOD("sparse", fallback="derivimplicit")
```

Supported aliases include:

```text
"linearimplicit"
"linear_implicit"
"implicitlinear"
"implicit_linear"
"affineimplicit"
"affine_implicit"
"sparse"
"be_linear"
"backward_euler_linear"
"linear_be"
```


### `"rosenbrock"`

A general linearly implicit method for nonlinear ODE systems.

Given:

```text
x' = f(x)
```

Rosenbrock-Euler evaluates the RHS and Jacobian at the beginning of the timestep, then solves:

```text
(I - gamma*dt*J) * k = dt*f(x)
x_next = x + damping*k
```

Example:

```python
class NonlinearState(S):
    S.STATE("x", "y")
    S.METHOD("rosenbrock", fallback="derivimplicit")

    S.DERIVATIVE(
        "x' = -a*x + y*y",
        "y' = -b*y + x",
    )
```

This is a good general-purpose method for nonlinear systems when `cnexp`, `bufferimplicit`, and `linearimplicit` do not apply. It is usually much faster than `derivimplicit` because it uses one RHS evaluation, one Jacobian evaluation, and one linear solve, without Newton iterations or line search.

For affine-linear systems, `"rosenbrock"` with `gamma=1.0` gives the same update as backward Euler. For nonlinear systems, it is an approximation to fully implicit backward Euler based on a local linearization.

Common options:

```python
S.METHOD(
    "rosenbrock",
    gamma=1.0,
    damping=1.0,
    jacobian_regularization=0.0,
    strict=True,
    fallback=None,
)
```

Supported aliases include:

```text
"rosenbrock"
"rosenbrock1"
"rosenbrock_euler"
"linearlyimplicit"
"linearly_implicit"
"linearized_implicit"
"linearly_implicit_euler"
"linimplicit"
"semiimplicit"
"semi_implicit"
```


### `"derivimplicit"`

Fully implicit backward-Euler integration for general ODE systems using Newton-Raphson iteration.

Use this for nonlinear and coupled ODE systems when a more specialized method is not available or when a fully converged implicit solve is desired.

Example:

```python
class GeneralImplicitState(S):
    S.STATE("x", "y")
    S.METHOD(
        "derivimplicit",
        newton_iters=4,
        damping=1.0,
        line_search=False,
    )

    S.DERIVATIVE(
        "x' = -a*x + y*y",
        "y' = -b*y + sin(x)",
    )
```

`derivimplicit` constructs the backward-Euler residual:

```text
R(x_next) = x_next - x - dt*f(x_next)
```

and approximately solves:

```text
R(x_next) = 0
```

using Newton iterations.

This is the most general method, but it is also the most expensive. It may require multiple RHS evaluations, Jacobian evaluations, and linear solves per timestep. It can also have convergence issues for very stiff systems, poor initial guesses, or large timesteps.

Dendra attempts to generate analytic Jacobians when possible. If analytic Jacobian generation fails, the method may fall back to a slower Jacobian path, depending on solver options.

Common options:

```python
S.METHOD(
    "derivimplicit",
    max_iter=8,
    tol=1e-8,
    damping=1.0,
    line_search=True,
    max_ls_steps=8,
    ls_decay=0.5,
    jacobian_regularization=0.0,
    detach_newton=None,
    create_graph=None,
)
```

Aliases for `max_iter` include:

```text
"newton_iters"
"iterations"
```

For compiled training workloads, a fixed small number of Newton iterations with line search disabled is often more compiler-friendly:

```python
S.METHOD(
    "derivimplicit",
    newton_iters=3,
    damping=1.0,
    line_search=False,
)
```

Supported aliases include:

```text
"derivimplicit"
"deriv_implicit"
"implicit"
"backward_euler"
"be"
```


### Choosing a method

A useful rule of thumb is:

- Use "cnexp" when each state equation is independent and affine in its own state.

- Use "bufferimplicit" for free/bound buffer systems such as oc/cai calcium buffering.

- Use "linearimplicit" or "sparse" for coupled affine-linear systems, such as Markov state models.

- Use "rosenbrock" for nonlinear systems when you want a fast, stable, fixed-work method.

- Use "derivimplicit" when you need the most general fully implicit solver or as a reference/fallback.

For example:

```python
# Independent HH-style gate
S.METHOD("cnexp")

# Calcium buffer motif
S.METHOD("bufferimplicit", bound="oc", free="cai")

# Coupled linear Markov model
S.METHOD("sparse")

# General nonlinear fast fallback
S.METHOD("rosenbrock", fallback="derivimplicit")

# Fully implicit Newton solve
S.METHOD("derivimplicit", newton_iters=4, line_search=False)
```

A recommended fallback pattern for nonlinear mechanisms is:

```python
S.METHOD("rosenbrock", fallback="derivimplicit")
```

A recommended fallback pattern for systems expected to be linear is:

```python
S.METHOD("sparse", fallback="rosenbrock")
```

This lets Dendra use the fastest valid solver when the symbolic structure matches, while retaining a more general method when it does not.


## Runtime noise and State SDEs

Dendra distinguishes **noise variables in mechanisms** from **stochastic differential equations in state variables**.

### Detached runtime noise for mechanisms

`Mechanism.GLOBALNOISE`, `Mechanism.BATCHNOISE`, and `Mechanism.RANGENOISE` declare buffers that are resampled during simulation. These are useful for stochastic currents or other exogenous drives:

```python
from dendra.models.mechanisms import Mechanism

class NoisyCurrent(Mechanism):
    Mechanism.RANGENOISE("eta", distribution="normal", mu=0.0, sigma=1.0)
    Mechanism.RANGE(sigma_i=0.01)

    def i(self, v):
        return self.sigma_i * self.eta
```

Runtime `NOISE` buffers are updated in-place under `torch.no_grad()` for speed and stable tensor identity. They are therefore **not differentiable with respect to their distribution parameters**. Gradients can still flow through ordinary coefficients such as `sigma_i` in the example above.

### First-class State SDEs

For stochastic dynamics of state variables, use `State.DIFFUSION(...)` with an SDE-aware `State.METHOD(...)`. Dendra currently provides two explicit State SDE methods:

- `State.METHOD("euler_maruyama")` for **Itô** SDEs.
- `State.METHOD("euler_heun")` for **Stratonovich** SDEs.

Both represent equations with drift and diffusion terms:

```text
dx = f(x, t) dt + g(x, t) dW
```

For `euler_heun`, the stochastic term is interpreted in Stratonovich form, often written with a circle:

```text
dx = f(x, t) dt + g(x, t) ∘ dW
```

Example:

```python
from dendra.models.mechanisms import State

class OU(State):
    State.STATE("x")
    State.RANGE(mu=0.0, tau=10.0, sigma=1.0)
    State.ASSIGNED("drift_mu", "drift_tau", "diff_sigma")

    State.DERIVATIVE("x' = (drift_mu - x) / drift_tau")
    State.DIFFUSION("x = diff_sigma")
    State.METHOD("euler_maruyama")

    def breakpoint(self, v, states):
        return {
            "drift_mu": self.mu,
            "drift_tau": self.tau,
            "diff_sigma": self.sigma,
        }
```

The generated Euler-Maruyama update is:

```text
x_next = x + f(x) dt + g(x) sqrt(dt) eps
```

where `eps` is a detached standard-normal increment.

For Stratonovich multiplicative noise, use Euler-Heun:

```python
class MultiplicativeNoise(State):
    State.STATE("x")
    State.RANGE(sigma=0.2)
    State.ASSIGNED("diff_sigma")

    State.DERIVATIVE("x' = 0.0")
    State.DIFFUSION("x = diff_sigma * x")
    State.METHOD("euler_heun")

    def breakpoint(self, v, states):
        return {"diff_sigma": self.sigma}
```

Euler-Heun evaluates a predictor state and averages the old/predicted diffusion coefficients:

```text
x_pred = x + f(x) dt + g(x) dW
x_next = x + f(x) dt + 0.5 * (g(x) + g(x_pred)) * dW
```

Pass `average_drift=True` to also average the deterministic drift term:

```python
State.METHOD("euler_heun", average_drift=True)
```

This v1 implementation is limited to `State` variables. It does not add noise directly to the voltage equation or to Dendra's specialized cable/voltage solvers.


## Q10 scaling
- Set `has_q10 = True` on a `State` subclass to enable temperature scaling. Dendra then exposes `self.q10()` in the state, calling `calc_q10()` if provided (otherwise defaulting to 1).
- Implement `calc_q10(self)` to return the multiplicative factor as a function of `self.celsius`. In the HH gates above:
  ```python
  def calc_q10(self):
      return 3.0 ** ((self.celsius - 6.3) / 10.0)
  ```
- Use `self.q10()` inside `breakpoint` or kinetic rate calculations to scale time constants and rates.


## State `inf` helper
- Implement `inf(self, v)` on a `State` to return a mapping of steady-state values for the declared state variables (e.g., `{"m": minf, "h": hinf}`) at a given voltage. Dendra uses this during initialization to set state variables unless overridden.
- You can also use `inf` to encode any custom initialization rule you want; it’s invoked during `initialize()` for states.


## Multiple State bundles per Mechanism
A mechanism can register multiple independent `State` subclasses. For example, the HH gates `m`, `h`, and `n` could each be defined as separate `State` classes and then registered via multiple `M.STATE(...)` calls (or a single call with multiple states). Dendra will aggregate all declared states/derivatives and step them together.



## Buffers and `initial`
- `BUFFER(name, ...)` declares auxiliary per-compartment buffers stored on the state (not evolved by an ODE). They are useful for geometry- or configuration-dependent terms reused in derivatives.
- Implement `initial(self, v)` on a `State` to populate these buffers (and optionally states) at initialization. It runs once during `initialize()` and can depend on morphology (e.g., `self.diam`).
- Example (extracellular Ca accumulation in `caextscale`):
  ```python
  class cao(S):
      S.STATE("cao")
      S.GLOBAL(lseg=1.0, txfer=4511.0, FARADAY=96500.0, cabath=2.0, fhspace=1.0)
      S.BUFFER("SA", "Vol_peri")
      S.DERIVATIVE("cao' = (ica*SA/(2*Vol_peri*FARADAY) + (cabath - cao)/txfer)")

      def initial(self, v):
          self.SA = math.pi * (1e-4) * self.diam * self.lseg
          Vol = math.pi * ((1e-4) * (self.diam / 2)) ** 2 * self.lseg
          self.Vol_peri = (
              math.pi * ((1e-4) * ((self.diam + self.fhspace) / 2)) ** 2 * self.lseg
          ) - Vol
  ```
  `SA` and `Vol_peri` are computed once, then reused inside the derivative during simulation.


## API overview
### State classmethods (at `State` definition)
- `STATE(name, ...)`: declare state variables.
- `METHOD(method, **kwargs)`: select the integration method and pass method-specific options. Examples include `"cnexp"`, `"bufferimplicit"`, `"linearimplicit"` / `"sparse"`, `"rosenbrock"`, and `"derivimplicit"`.
- `DERIVATIVE("x' = expr", ...)`: symbolic ODEs for states.
- `KINETIC("~ a <-> b (alpha, beta)", ...)`: Markov/kinetic schemes between states.
- `ASSIGNED(name, ...)`: computed per-compartment variables; set in `breakpoint`.
- `GLOBAL` / `RANGE`: shared vs per-compartment parameters scoped to the State.
- `BUFFER(name, ...)`: declare auxiliary per-compartment buffers initialized in `initial` and reused in derivatives/assigned calculations.

`State` dynamics are local to one mechanism. When a variable needs to be shared across mechanisms or spatially processed over the population, represent it as a `Material` instead of a private `State`.



### State lifecycle hooks
- `initial(self, v)`: optional initializer run during `initialize()` to set buffers/states (e.g., geometry-dependent buffers declared via `BUFFER`).
- `breakpoint(self, v, states=None)`: compute `ASSIGNED`/intermediate values each step before derivatives/currents are evaluated; return a dict matching `ASSIGNED` names if you choose to return values.
- `inf(self, v)`: return steady-state values for state variables (used for initialization unless overridden by the caller).
- `calc_q10(self)`: optional helper when `has_q10=True`; returns the temperature scaling factor consumed by `self.q10()` inside your kinetics.


### Mechanism classmethods (at mechanism definition)
- `STATE(StateSubclass, ...)`: register State bundles used by the mechanism.
- `GLOBAL`, `RANGE`: shared vs per-compartment parameters on the mechanism.
- `ASSIGNED(name, ...)`: per-compartment buffers/computed vars (analogous to `BUFFER` in State); typically set in `initial`/`breakpoint`.
- `INIT(...)`: initialization overrides for mechanism/state buffers.
- `USEION(ion, read=(), write=())`: declare ionic dependencies. `read`/`write` must be in `{ion}i`, `{ion}o`, `e{ion}`, `i{ion}`. For example, read `("ena", "nai", "nao")` and write `("ina",)` to contribute Na current.
- `USEMATERIAL(material, read=(), write=(), source=())`: declare generic shared-material dependencies. Use this for non-ion shared fields such as `ip3i`, `campi`, `atpi`, `pip2`, mobile buffers, or for ion-backed fields when you want material-style access rather than current assembly.
- `NONSPECIFIC_CURRENT(name, ...)`: register membrane currents not tied to an ion.
- `EXPLICIT(name, ...)`: mark voltage-independent currents when assembling conductance terms.

For full-field material dynamics, use a `MaterialProcess` subclass instead of an ordinary `Mechanism`. Process subclasses also support declaration helpers such as `PHASE(...)`, `METHOD(...)`, and process-specific declarations like `DIFFUSE(...)`, `CLEAR(...)`, `CLAMP(...)`, or `EXCHANGE(...)`.



### Mechanism lifecycle hooks
- `initial(self, v)`: optional initializer run during `initialize()`; set mechanism-level `ASSIGNED` buffers or other cached quantities.
- `breakpoint(self, v)`: compute mechanism currents and mechanism-level `ASSIGNED` values each step (called before current assembly).
- Current methods (`ina`, `ik`, `il`, etc.): implement currents declared via `NONSPECIFIC_CURRENT` or ion writes in `USEION`. For an ordinary distributed `Mechanism`, they return outward-positive mA/cm² and their voltage derivative is S/cm². A `PointProcess` instead returns lumped nA and uses µS conductances; Dendra performs the area normalization.



**Example (h-channel with mechanism-level ASSIGNED):**
```python
class h(M):
    M.STATE(s, f)
    M.GLOBAL(gbar=0.0001)  # S/cm² (distributed conductance density)
    M.USEION("k", read=["ek"], write=["ik"])
    M.USEION("na", read=["ena"], write=["ina"])
    M.ASSIGNED("g")  # mechanism-level buffer, set in initial/breakpoint

    def initial(self, v):
        self.breakpoint(v)  # compute g at t=0

    def breakpoint(self, v):
        self.g = self.gbar * (0.5 * self.s + 0.5 * self.f)

    def ina(self, v):
        return 0.5 * self.g * (v - self.ena)

    def ik(self, v):
        return 0.5 * self.g * (v - self.ek)
```
Here `ASSIGNED("g")` acts like a mechanism-level buffer (analogous to `BUFFER` in State) that is populated in `initial`/`breakpoint` and reused in current computations. Because `h` derives from ordinary `Mechanism`, `gbar` and `g` are conductance densities in S/cm², `v`/`ena`/`ek` are in mV, and `ina`/`ik` return mA/cm².


## Ions
Dendra has built-in support for ionic concentrations and currents via the `USEION` mechanism classmethod. `USEION` is the right interface when a mechanism contributes an ionic current, reads an ionic reversal potential, or updates an ionic concentration.

Built-in ions and their standard fields are:

- Sodium (`"na"`): `nai`, `nao`, `ena`, `ina`
- Potassium (`"k"`): `ki`, `ko`, `ek`, `ik`
- Calcium (`"ca"`): `cai`, `cao`, `eca`, `ica`

The ion-field convention is fixed: `<ion>i` and `<ion>o` concentrations are in mM, `e<ion>` is in mV, and the assembled `i<ion>` field is outward-positive current density in mA/cm². A distributed mechanism that writes an ionic current therefore returns mA/cm²; a point process authors its lumped current in nA and Dendra normalizes that contribution before ion-current assembly.

Ions are also specialized `Material` objects. This means ion concentration fields such as `cai` can participate in generic material processes such as diffusion, while still retaining ion-specific behavior such as valence, ionic current bookkeeping, and Nernst/reversal-potential updates.

You can define custom ions with `register_ion`:

```python
from dendra.models.mechanisms import register_ion

register_ion(
    "cl",
    valence=-1,
    e=-65.0,     # reversal potential [mV]
    i0=10.0,   # intracellular concentration [mM]
    o0=110.0,  # extracellular concentration [mM]
)
```

This makes `"cl"` available in `USEION` declarations with fields `cli`, `clo`, `ecl`, and `icl`.

### USEION read/write semantics

`USEION` declares that a mechanism interacts with ion-specific variables. For example:

```python
M.USEION("na", read=["ena", "nai", "nao"], write=["ina"])
```

means the mechanism reads sodium reversal potential and concentrations and contributes sodium current. Dendra then makes `ena`, `nai`, and `nao` available to the mechanism and its `State` bundles, and expects the mechanism to define an `ina(self, v)` current method.

Concentration variables `<ion>i` and `<ion>o` can also be written by mechanisms when implementing local ion dynamics. For example:

```python
M.USEION("ca", read=["ica"], write=["cai"])
```

makes `ica` available to the state dynamics and commits the local updated `cai` back to the population-wide calcium material/ion field. Dendra does **not** automatically turn membrane current into a concentration derivative: the model must supply the Faraday, valence, area, and volume conversion explicitly, as in the following example.

### Example: intracellular Ca²⁺ accumulation with buffering

```python
import math

from dendra.models.mechanisms import Mechanism as M
from dendra.models.mechanisms import State as S
from dendra.models.mechanisms.ops import *


class oc_cai(S):
    S.STATE("oc", "cai")
    S.METHOD("bufferimplicit", bound="oc", free="cai")
    S.GLOBAL_SIGNED(lseg=1e-3, ku=100.0, kr=0.238, nb=4.0, Bi=0.001, FARADAY=96500.0)
    S.BUFFER("SA", "Vol")

    S.DERIVATIVE(
        "oc' = ku * cai * (1-oc) - kr * oc",
        "cai' = (-ica * SA / Vol / (2*FARADAY) - nb * Bi * (ku * cai * (1 - oc) - kr * oc))",
    )

    def initial(self, v):
        self.SA = math.pi * (1e-4) * self.diam * self.lseg
        self.Vol = math.pi * ((1e-4) * (self.diam / 2)) ** 2 * self.lseg


class caintscale(M):
    M.STATE(oc_cai)
    M.USEION("ca", read=["ica"], write=["cai"])
    M.INIT(oc=0.05)
```

How this works:

- `oc_cai` declares private local state variables `oc` and `cai`.
- `S.METHOD("bufferimplicit", bound="oc", free="cai")` uses the specialized implicit calcium-buffer update, avoiding a generic Newton solve while preserving the free/bound conservation relation under the backward-Euler discretization.
- `M.USEION("ca", read=["ica"], write=["cai"])` tells Dendra that the mechanism uses the local calcium current to update intracellular calcium.
- The updated `cai` is committed back to the shared population-wide calcium field, so other mechanisms and material processes can read it later in the timestep schedule.

Use `USEION` for ion-specific electrophysiology. Use `USEMATERIAL` when the shared variable is not an ion, or when you want generic material-style access to an ion-backed field.


## Materials: generic shared state

`Material` generalizes the ion pattern to arbitrary population-wide fields. Materials are useful when a variable:

- exists across all compartments of a population,
- may be read by several mechanisms,
- may be written or sourced by one or more mechanisms,
- may have spatial/global dynamics such as diffusion, clearance, clamping, or exchange,
- should be inspectable as shared model state rather than private state inside one mechanism.

Examples include `ip3i`, `campi`, `atpi`, `pip2`, mobile buffers, extracellular transmitter concentration, metabolic variables, pH/proton concentration, store variables such as `caer`, or ion concentration fields accessed through the material interface.

A generic material can be registered globally:

```python
from dendra.models.mechanisms import register_material

register_material(
    "ip3",
    fields={"ip3i": 0.1},
    min_values={"ip3i": 0.0},
    domain={"ip3i": "intracellular"},
    units={"ip3i": "mM"},
    conserved={"ip3i": True},
)
```

or configured on a population/model:

```python
self.material(
    "ip3",
    fields={"ip3i": 0.1},
    min_values={"ip3i": 0.0},
    domain={"ip3i": "intracellular"},
    units={"ip3i": "mM"},
    conserved={"ip3i": True},
)
```

`domain`, `units`, and `conserved` are field-level metadata and may each be one value for every field or a per-field mapping. Common domains are `"intracellular"` / `"i"`, `"extracellular"` / `"o"`, and `"membrane"`; geometry-aware material processes use the domain to choose an appropriate volume or area. The `units` string is descriptive and inspectable through `material.field_spec(field).units`: Dendra does not convert values or perform dimensional analysis. The `conserved` flag likewise records intent but does not itself enforce conservation; use a conservation-preserving process and compatible geometry/units when conservation matters.

Generic materials have no universal default unit. Every initial value, mechanism read/write, additive source, clamp target, and process parameter must follow the convention declared by the model. Concentration-like fields commonly use mM, but dimensionless gates, molecule counts, surface densities, and other conventions are also valid when documented consistently.

### Initial values, parameters, and runtime state

Each generic material field has a registered **initial source** and a separate mutable runtime tensor. A plain number or tensor is registered as a fixed parameter. Pass a `torch.nn.Parameter`, a Dendra parametric module, or create the model inside `dn.ctx(REQUIRE_GRAD=1)` when the initial field should be optimized. For example:

```python
initial_ip3 = torch.nn.Parameter(torch.tensor(0.1))
self.material("ip3", fields={"ip3i": initial_ip3})
```

After the model is built, `model.mech.materials["ip3"].initial_source("ip3i")` returns that registered source. `model.initialize()` resolves it afresh: in training mode the live field retains the autograd path through subsequent mechanisms and material processes, while evaluation mode detaches mutable simulation state. A normal PyTorch `state_dict` contains both model inputs and the current field. Dendra's runtime checkpoint helpers restore the mutable field without overwriting the initial source, so resuming a trajectory does not silently change fitted parameters.

Ions are specialized materials. For example, `"ca"` owns `cai`, `cao`, `ica`, and `eca`, and can be accessed through `USEION` for electrophysiology or through `USEMATERIAL` for generic material processes. Ions retain their dedicated `e_init`, `i_init`, and `o_init` parameters rather than duplicating these generic initial sources.


## `USEMATERIAL`: mechanism access to shared fields

Mechanisms declare generic shared-field dependencies with `M.USEMATERIAL(...)`.

### Read-only material access

```python
class ip3_sensor(M):
    M.USEMATERIAL("ip3", read=["ip3i"])
    M.RANGE(kd=0.2)
    M.ASSIGNED("activation")

    def breakpoint(self, v):
        self.activation = self.ip3i / (self.ip3i + self.kd)
```

`ip3i` is exposed as a local view of the population-wide material field, using the same placement/indexing logic as ion reads.

### Replacement writes

Use `write=[...]` when one local mechanism owns the local update of a material field:

```python
class ip3_decay_state(S):
    S.STATE("ip3i")
    S.METHOD("cnexp")
    S.DERIVATIVE("ip3i' = -ip3i / tau_ip3")


class ip3_decay(M):
    M.USEMATERIAL("ip3", read=["ip3i"], write=["ip3i"])
    M.STATE(ip3_decay_state)
    M.RANGE(tau_ip3=1000.0)  # ms
```

The mechanism reads the start-of-step local `ip3i`, advances it locally, and commits the updated value back to the shared material field during the material commit phase.

### Additive sources

Use `source={field: local_buffer}` when mechanisms contribute additive increments to a shared field:

```python
class ip3_production(M):
    M.USEMATERIAL("ip3", source={"ip3i": "dip3"})
    M.RANGE(production=0.001)  # mM per committed increment
    M.ASSIGNED("dip3")

    def breakpoint(self, v):
        self.dip3 = self.production
```

Material-source semantics are additive **increments** in the target field's units, committed after local mechanisms run. For the mM `ip3i` declaration above, `dip3` must therefore be in mM per committed increment. If a model instead starts from a rate in mM/ms, multiply it by `dt` (ms) explicitly in the mechanism or implement a process with rate semantics.

### Barriered shared-state semantics

Material reads and writes are scheduled to avoid hidden ordering effects:

1. mechanisms read a consistent material snapshot,
2. local mechanisms compute states, currents, writes, and sources,
3. material writes/sources are committed together,
4. material processes such as diffusion operate on the committed full field,
5. materials/ions apply post-processing such as minimum-value guarding and Nernst updates.

This is intentionally different from one mechanism directly reaching into another mechanism's private buffers.


## MaterialProcess: population-field dynamics

`MaterialProcess` is the process-level analogue of `Mechanism` for population-wide material fields. It is inserted like a mechanism, binds the shared material tensors, and is scheduled in the material phase. Processes may be global; spatial processes such as diffusion may also be restricted to selected compartments. This is the right place for spatial/global dynamics such as diffusion, clearance, bath clamping, or conservative exchange between stores.

Material processes support:

- `MaterialProcess.USEMATERIAL(...)`: declare full-field material dependencies.
- `MaterialProcess.PHASE(...)`: choose where the process runs in the material scheduler.
- `MaterialProcess.METHOD(method, **kwargs)`: choose process-specific numerical behavior.

Current material phases are:

```text
post_local      local pointwise material processes; good for clearance/exchange
transport       spatial transport; good for diffusion
post_transport  clamps, bath conditions, bounds, and boundary constraints
```

The scheduler first commits local mechanism material writes/sources, then runs material processes by phase, and finally advances materials/ions to apply guarding and derived-field updates such as Nernst reversal potentials.

### `DiffusionProcess`

`DiffusionProcess` implements finite-volume material diffusion. The current implementation supports intracellular/cytosolic fields on unbranched 1D axons with analytic geometry, branched `Tree` morphologies with native material geometry, and explicit named chemical geometries on both unbranched and `Tree` populations (including intracellular, extracellular, or periaxonal domains).

#### Regional support and topology

A process can be inserted globally or on any compartment subset. A restricted insertion uses the induced morphology graph: an edge transports material only when both endpoints belong to the process region. Edges crossing out of the region are sealed, disconnected selected components evolve as independent matrix blocks, and excluded field values remain exactly unchanged. The selected subset does not need to be connected.

Generic intracellular tracer diffusion:

```python
from dendra.models.mechanisms import DiffusionProcess


class tracer_diffusion(DiffusionProcess):
    DiffusionProcess.PHASE("transport")
    DiffusionProcess.USEMATERIAL("tracer", read=["traceri"], write=["traceri"])
    DiffusionProcess.RANGE(Dtracer=0.5)  # µm²/ms
    DiffusionProcess.DIFFUSE("tracer", field="traceri", D="Dtracer")
    DiffusionProcess.METHOD("implicit", solver="auto", boundary="sealed")
```

Then register the material and insert the process:

```python
self.material(
    "tracer",
    fields={"traceri": 0.0},
    min_values={"traceri": 0.0},
    domain={"traceri": "intracellular"},
    units={"traceri": "mM"},
    conserved={"traceri": True},
)
self.insert(tracer_diffusion, Dtracer=0.5)
```

To constrain transport to a region, insert on a slice instead:

```python
self.dendrites.insert(tracer_diffusion, Dtracer=0.5)
```

Repeated insertions of the same process class form one union domain. Adjacent selections therefore communicate even when they came from separate insertion calls, while disjoint selections remain independent. Overlapping diffusivity assignments must agree. Distinct diffusion-process classes may target disjoint regions, but cannot overlap when they write the same material field in the same phase because sequential implicit solves would be order-dependent.

#### Named chemical geometry and edge diffusivity

For a chemical domain that does not use the model's native geometry, register explicit finite-volume storage and edge geometry. This registry is independent of electrical extracellular layers: `ExtCellAxon.xraxial`, `xc`, and `xg` do not define a chemical extracellular volume or diffusion area. Quantities may be scalar or broadcastable tensors. Node quantities normally end in `C`. Unbranched edge quantities end in `C - 1`; Tree edge quantities end in compact dimension `E` and use `tree.material_edge_index` order.

The example assumes that `k.ko` already exists in the Material/Ion registry with extracellular domain metadata (normally because the model uses the `k` Ion), and that a Slice named `periaxonal_region` identifies the desired transport support. Every field's effective domain must match the named geometry's domain.

```python
# This unbranched example uses (N, C) nodes and (N, C - 1) edges.
self.register_buffer("periaxonal_volume", volume_um3)
self.register_buffer("periaxonal_interface_area", area_um2)
self.register_buffer("periaxonal_interface_distance", distance_um)
self.register_buffer("edge_D", edge_diffusivity_um2_per_ms)

self.register_material_geometry(
    "periaxonal",
    domain="extracellular",
    volume="periaxonal_volume",
    edge_area="periaxonal_interface_area",
    edge_distance="periaxonal_interface_distance",
)

class periaxonal_k_transport(DiffusionProcess):
    DiffusionProcess.RANGE(bath_rate=0.0)
    DiffusionProcess.GLOBAL(bath_target=5.0)
    DiffusionProcess.DIFFUSE(
        "k", field="ko", D="edge_D", domain="extracellular",
        geometry="periaxonal", D_location="edge",
    )
    DiffusionProcess.RELAX(
        "k", field="ko", rate="bath_rate", target="bath_target",
        where="all",
    )
    DiffusionProcess.METHOD("implicit")

# bath_rate_on_region is node-aligned on this Slice and is nonzero only
# where exchange with the fixed bath should occur.
self.periaxonal_region.insert(
    periaxonal_k_transport, bath_rate=bath_rate_on_region,
)
```

For a `Tree`, the morphology still owns adjacency while named geometry supplies chemical storage and weights. Tree edge tensors use a compact, stable `E` axis rather than a root-padded `C` axis:

```python
parent, child = tree.material_edges
edge_index = tree.material_edge_index  # shape (2, E)

# edge_factor[e] and edge_D[e] describe parent[e] -> child[e].
tree.register_buffer("ecs_volume", volume_um3)       # (..., C)
tree.register_buffer("ecs_edge_factor", factor_um)  # (..., E)
tree.register_buffer("ecs_edge_D", edge_D)          # (..., E)
tree.register_material_geometry(
    "ecs_tree", domain="extracellular",
    volume="ecs_volume", edge_factor="ecs_edge_factor",
)

class tree_ecs_transport(DiffusionProcess):
    DiffusionProcess.DIFFUSE(
        "k", field="ko", D="ecs_edge_D",
        domain="extracellular", geometry="ecs_tree",
        D_location="edge",
    )
    DiffusionProcess.METHOD("implicit")
```

Tree edge order is increasing child storage index with the morphology root omitted. Named geometry can set an existing edge factor or diffusivity to zero to seal that edge, but it cannot add an edge. Regional selection remains exact: omitting an intervening junction seals its incident crossing edges rather than silently adding the junction to the process region.

`volume` is in µm³, `edge_area` in µm², `edge_distance` in µm, and the equivalent precomputed `edge_factor = edge_area / edge_distance` in µm. Edge conductance is `D_edge * edge_factor` in µm³/ms. With named geometry and the default `D_location="node"`, `D` is scalar or node-aligned and the interface value is the arithmetic endpoint mean `0.5 * (D_i + D_j)`. Named geometry rejects `interface_scheme="series"` because its area/distance edge factor does not contain the two half-compartment resistances.

With `D_location="edge"`, `D` is scalar or interface-aligned; a string such as `"edge_D"` can name a registered Population edge buffer or parameter. Edge-located D requires named explicit geometry and is available on unbranched and Tree populations, but not with analytic cylindrical 1D or native Tree geometry. A node-aligned `RANGE` parameter is not an edge array. Process-local parameter/buffer names take precedence over Population names.

String geometry components and Population coefficient names must refer to registered buffers or parameters. They therefore follow dtype/device moves and state dictionaries. Differentiable sources preserve autograd. Direct ordinary tensor-like geometry arguments are registered as Population buffers automatically, while a direct `nn.Parameter` is registered as a Population parameter and is discoverable through `model.parameters()`.

Geometry, D, and RELAX rate/target/where values are sampled when the spatial operator is configured—normally on the first step after initialization or after explicit timestep reconfiguration—not read anew on every timestep. Mutating one during a run requires reconfiguring before it takes effect; this declaration API does not represent a continuously time-varying bath coefficient.

On active process support, unbranched volumes must be finite and positive. Implicit Tree diffusion permits finite zero-volume algebraic junctions when every conductive component reaches at least one positive-volume node; explicit Tree diffusion requires all active volumes to be positive. Interface area/factor and D must be finite and non-negative, and explicit edge distance finite and positive. A zero edge factor or zero edge D seals that interface. Validation that depends on the selected region occurs when the process binds or its operator is configured.

A process class can diffuse several fields through one named geometry. If any `DIFFUSE` declaration names a geometry, every `DIFFUSE` declaration in that class must name that same geometry; named and unnamed declarations cannot be mixed. On a Tree, named geometry replaces chemical volumes and edge weights but retains morphology adjacency. It may seal existing edges, but cannot add connections or define a chemically distinct graph.

#### Fused reservoir or bath coupling

`RELAX` (alias `BATH`) contributes `-rate * (c - target)` directly to the same backward-Euler system as diffusion. If `M` is finite-volume storage, `L` is the diffusion Laplacian, `Lambda` is the effective relaxation rate, and `c_source` is the field after committed local Mechanism writes/sources, Dendra solves

```text
(M + dt*L + dt*M*Lambda) c_new
    = M*c_source + dt*M*Lambda*c_bath
```

Diffusion and bath exchange are therefore not split from one another, and stiff non-negative bath rates remain stable. `RELAX` is implicit-only and must target a field declared by `DIFFUSE` in the same class. To model bath coupling without longitudinal diffusion, declare `DIFFUSE(..., D=0)` and add `RELAX(...)`. Multiple `RELAX` declarations for one field add their matrix contributions.

The effective relaxation support is

```text
process insertion region ∩ where mask ∩ compartments with nonzero rate
```

`where="all"` (the default) or `where=None` means **no additional mask**. It never bypasses the region where the DiffusionProcess was inserted. A boolean scalar/tensor, a node-aligned process parameter/buffer name, or a registered Population buffer/parameter name can further restrict coupling. Process-local names take precedence. No other string is special: ClampProcess selectors such as `"terminal"`, `"root"`, or `"soma"` are treated as quantity names here, not topology selectors.

Setting a node-aligned rate to zero is often the simplest independent bath mask. In the example above, a regional process with `where="all"` still excludes every compartment outside that region, and only nodes with `bath_rate != 0` exchange with the bath. `rate` is in ms⁻¹ and must be finite and non-negative on effective support; `target` has the field's units and must be finite wherever the rate is nonzero.

`RELAX` couples the modeled field to a fixed/infinite reservoir and therefore does not conserve mass inside the modeled domain. Use `ExchangeProcess` instead when the second reservoir is finite, explicitly modeled, and total mass must be conserved.

Material scheduling is: accepted local current/material sources → transport processes such as this fused solve → Material/Ion guards and derived-field refresh (including enabled Nernst updates).

Intracellular Ca²⁺ diffusion through the ion-as-material bridge:

```python
class cai_diffusion(DiffusionProcess):
    DiffusionProcess.PHASE("transport")
    DiffusionProcess.USEMATERIAL("ca", read=["cai"], write=["cai"])
    DiffusionProcess.RANGE(Dca=0.6)  # µm²/ms
    DiffusionProcess.DIFFUSE("ca", field="cai", D="Dca")
    DiffusionProcess.METHOD("implicit", solver="auto", boundary="sealed")
```

This updates the same population-wide `cai` field used by `USEION("ca", ...)` mechanisms. Local calcium mechanisms can update `cai`, diffusion spreads the committed field, and then the calcium ion object can update/guard `cai` and recompute `eca` if appropriate.

Solver and geometry notes:

- `METHOD("implicit")` uses finite-volume backward Euler and is the recommended default for diffusion.
- `solver="auto"` selects the appropriate spatial backend when available.
- For Dendra's unbranched 1D `Axon` classes, diffusion uses analytic compartment volume and 1D tridiagonal/Thomas-style solves.
- Named unbranched and Tree chemical geometries use explicit control volumes and either `edge_factor` or `edge_area / edge_distance`. They accept the arithmetic interface scheme; use `D_location="edge"` when the desired interface coefficient is already known.
- `RELAX(...)` is implicit-only. Explicit diffusion remains available when no RELAX declaration is present. Tree RELAX uses the volume selected by the diffusion operator—native for unnamed diffusion and custom for named geometry.
- Analytic 1D diffusion uses the legacy arithmetic endpoint interface by default. Pass `interface_scheme="series"` to `METHOD(...)` to combine the two half-compartment diffusion resistances in series. The latter is useful for NEURON RxD parity across sharp diameter or length changes. Tree diffusion already consumes precomputed edge geometry and does not accept this 1D-only option.
- For `Tree` morphologies imported through NEURON, diffusion uses tree/DHS-style topology when material geometry buffers such as `volume_i` and `diff_geom_um` are available.
- Diffusion coefficients `D` are finite, non-negative effective diffusivities in µm²/ms on active interfaces; they are not automatically temperature-scaled unless a process explicitly implements such scaling.
- Axon and Tree finite-volume geometry uses compartment volumes in µm³. Diffusion preserves the volume-weighted field total under sealed boundaries (up to solver/floating-point error); the field's `conserved=True` metadata documents this intent but does not activate a different algorithm.

### `ClearanceProcess`

`ClearanceProcess` applies pointwise first-order relaxation/degradation:

```text
c' = -rate * (c - target)
```

The default method is the exact exponential update for fixed `rate` and `target` over the timestep. `rate` is in ms⁻¹, while `target` has the same units as `c`.

```python
from dendra.models.mechanisms import ClearanceProcess


class ip3_clearance(ClearanceProcess):
    ClearanceProcess.PHASE("post_local")
    ClearanceProcess.USEMATERIAL("ip3", read=["ip3i"], write=["ip3i"])
    ClearanceProcess.RANGE(kclear=0.01)  # ms⁻¹
    ClearanceProcess.CLEAR("ip3", field="ip3i", rate="kclear", target=0.0)
    ClearanceProcess.METHOD("exact")
```

Useful aliases include:

```python
ClearanceProcess.CLEAR(...)
ClearanceProcess.DECAY(...)
ClearanceProcess.RELAX(...)
```

Use clearance for IP3/cAMP degradation, extracellular uptake, recovery to baseline, bath relaxation, or slow homeostatic removal.

### `ClampProcess`

`ClampProcess` imposes algebraic constraints on full material fields. It is useful for bath conditions, boundary conditions, fixed experimental fields, and safety bounds. It defaults to the `post_transport` phase, so by default it runs after diffusion/transport.

Hard clamp all extracellular potassium to a bath value:

```python
from dendra.models.mechanisms import ClampProcess


class ko_bath_clamp(ClampProcess):
    ClampProcess.USEMATERIAL("k", read=["ko"], write=["ko"])
    ClampProcess.RANGE(ko_bath=5.4)
    ClampProcess.CLAMP("k", field="ko", value="ko_bath", where="all")
```

Clamp only terminal/boundary compartments:

```python
class terminal_ko_bath(ClampProcess):
    ClampProcess.USEMATERIAL("k", read=["ko"], write=["ko"])
    ClampProcess.RANGE(ko_bath=5.4)
    ClampProcess.CLAMP("k", field="ko", value="ko_bath", where="terminal")
```

Bounds and positivity constraints:

```python
class ip3_nonnegative(ClampProcess):
    ClampProcess.USEMATERIAL("ip3", read=["ip3i"], write=["ip3i"])
    ClampProcess.MIN("ip3", field="ip3i", value=0.0)


class ip3_bounds(ClampProcess):
    ClampProcess.USEMATERIAL("ip3", read=["ip3i"], write=["ip3i"])
    ClampProcess.RANGE(ip3_max=10.0)
    ClampProcess.BOUNDS("ip3", field="ip3i", lower=0.0, upper="ip3_max")
```

Supported MVP locations include `"all"`, `"terminal"` / `"boundary"`, `"root"` / `"soma"`, boolean masks, and explicit index/slice-like selectors.

### `ExchangeProcess`

`ExchangeProcess` implements conservative pointwise exchange between two material fields:

```text
V_a * a' = -g * (a - b)
V_b * b' =  g * (a - b)
```

It preserves `V_a*a + V_b*b` up to floating-point roundoff when both pools have positive volume/mass. The two fields must use the same units because the process evaluates `a - b`. The default method is the exact closed-form update for fixed exchange coefficient and volumes over the timestep.

Example: cytosol ↔ ER-like calcium store exchange:

```python
from dendra.models.mechanisms import ExchangeProcess


class ca_er_exchange(ExchangeProcess):
    ExchangeProcess.USEMATERIAL("ca", read=["cai"], write=["cai"])
    ExchangeProcess.USEMATERIAL("er_ca", read=["caer"], write=["caer"])
    ExchangeProcess.RANGE(k_er=1e-3, Ver=0.1)  # k_er: ms⁻¹; Ver: µm³

    ExchangeProcess.EXCHANGE(
        ("ca", "cai"),
        ("er_ca", "caer"),
        rate="k_er",
        volume_a="intracellular",
        volume_b="Ver",
    )
    ExchangeProcess.METHOD("exact")
```

Register the store material before insertion:

```python
self.material(
    "er_ca",
    fields={"caer": 0.2},
    min_values={"caer": 0.0},
    domain={"caer": "intracellular"},
    units={"caer": "mM"},
    conserved={"caer": True},
)
self.insert(ca_er_exchange, k_er=1e-3, Ver=0.1)
```

You can specify either `rate=...` or `conductance=...`:

- `rate` has units ms⁻¹; Dendra converts it to an exchange conductance with `g = rate * volume_a`.
- `conductance` is used directly and must have units of the chosen volume/mass weight per ms (µm³/ms when using inferred volumetric geometry).
- Inferred intracellular/extracellular volume weights are in µm³; inferred membrane/surface weights are in µm². Explicit scalar/tensor weights must follow one compatible convention for both pools. Setting material metadata `conserved=True` records intent but does not alter these equations.

Useful aliases include:

```python
ExchangeProcess.EXCHANGE(...)
ExchangeProcess.COUPLE(...)
ExchangeProcess.TRANSFER(...)
```


## Choosing between State, Ion, Material, and MaterialProcess

Use **private `State` variables** when the variable belongs to one mechanism and no other mechanism needs to see it. Examples: HH gates, Markov states, private kinetic intermediates.

Use **`USEION`** when the mechanism participates in electrophysiological ion handling: reading `ena`/`ek`/`eca`, reading ion concentrations, writing ionic currents such as `ina`, or locally updating ion concentrations such as `cai`.

Use **`USEMATERIAL`** when the mechanism needs to share a generic chemical/biochemical field with other mechanisms. Examples: IP3, ATP, cAMP, PIP2, pH/protons, mobile buffers, ligand concentration, extracellular transmitter, or ion-backed concentration fields used as generic shared state.

Use **`MaterialProcess`** when the dynamics are naturally full-field or spatial/global rather than private to one local mechanism.

Current concrete process families:

- **`DiffusionProcess`**: finite-volume spatial diffusion of material fields. Use for intracellular Ca²⁺ diffusion, IP3 diffusion, mobile-buffer diffusion, etc.
- **`ClearanceProcess`**: first-order degradation, uptake, or relaxation to a target. Use for IP3/cAMP degradation, extracellular clearance, or recovery to baseline.
- **`ClampProcess`**: algebraic bath conditions, boundary clamps, or safety bounds. Use for fixed extracellular bath values, terminal boundary conditions, or nonnegative concentration constraints.
- **`ExchangeProcess`**: conservative exchange between two full material fields. Use for cytosol ↔ store exchange, mobile ↔ immobile pools, or finite reservoir exchange.

A compact mental model is:

- Mechanisms are local operators.
- States are private local memory.
- Materials are shared population-wide fields.
- MaterialProcesses are spatial/global operators on shared fields.
- Ions are specialized Materials with electrophysiological semantics.

For coupled material models, compose local mechanisms and material processes through explicit shared fields rather than by directly reaching into another mechanism's private buffers.


## Example material-process composition

The material interface is most useful when several mechanisms and processes share one field. For example, a calcium model might combine:

```python
class caintscale(M):
    M.STATE(oc_cai)                     # local buffer/reaction dynamics
    M.USEION("ca", read=["ica"], write=["cai"])

class cai_diffusion(DiffusionProcess):
    DiffusionProcess.PHASE("transport")
    DiffusionProcess.USEMATERIAL("ca", read=["cai"], write=["cai"])
    DiffusionProcess.RANGE(Dca=0.6)  # µm²/ms
    DiffusionProcess.DIFFUSE("ca", field="cai", D="Dca")
    DiffusionProcess.METHOD("implicit")

class cai_minimum(ClampProcess):
    ClampProcess.USEMATERIAL("ca", read=["cai"], write=["cai"])
    ClampProcess.MIN("ca", field="cai", value=1e-12)
```

The local mechanism updates `cai` from calcium current and buffering, diffusion spreads the committed `cai` field, and the clamp enforces a final lower bound. Other mechanisms can read the same `cai` field with either `USEION("ca", read=["cai"])` or `USEMATERIAL("ca", read=["cai"])`, depending on whether they need ion-specific electrophysiology or generic material-style access.
